# Training Metrics and Hardware Utilization Visualization

This notebook loads `metrics.json` and creates interactive Plotly charts to analyze model convergence and hardware utilization of your RTX 4070 GPU during training.

In [7]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [8]:
# Load the metrics data robustly
import os
metrics_file = "metrics.json"
if not os.path.exists(metrics_file):
    # Try relative path from workspace root
    metrics_file = os.path.join("chapter_1_transformers", "basic_transformer_implementation", "metrics.json")

with open(metrics_file, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df

,epoch,train_loss,val_ppl,val_acc,epoch_time_seconds,throughput_tokens_sec,goodput_tokens_sec,peak_vram_allocated_mb,peak_vram_reserved_mb,tflops_per_sec,mfu_percent
0,1,1.812591,6.196528,0.462061,111.832801,179513.701262,179513.701262,43.90918,60.0,0.509482,0.420018
1,2,1.689722,6.002355,0.470929,114.330725,175591.643600,175591.643600,43.90918,60.0,0.494420,0.407601
2,3,1.668022,5.921768,0.476541,113.353705,177105.106701,177105.106701,43.90918,60.0,0.499717,0.411968
3,4,1.656932,5.928221,0.477916,116.418800,172442.251842,172442.251842,43.90918,60.0,0.488597,0.402801
4,5,1.649679,5.934260,0.478984,112.275812,178805.387514,178805.387514,43.90918,60.0,0.505289,0.416562
5,6,1.644599,5.941358,0.480244,115.099357,174419.045010,174419.045010,43.90918,60.0,0.493240,0.406628
6,7,1.640296,5.939281,0.479436,115.251308,174189.085394,174189.085394,43.90918,60.0,0.494536,0.407697
7,8,1.637083,5.874669,0.481224,112.621692,178256.246215,178256.246215,43.90918,60.0,0.504264,0.415716
8,9,1.634133,5.901512,0.480696,113.868818,176303.929580,176303.929580,43.90918,60.0,0.498056,0.410598
9,10,1.631350,5.885869,0.482515,111.885612,179428.969498,179428.969498,43.90918,60.0,0.507748,0.418589


## 1. Machine Learning Metrics (Loss, Perplexity, Accuracy)

In [9]:
# Plot Train Loss and Dev Accuracy
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['train_loss'], name="Train Loss", mode="lines+markers", line=dict(color="royalblue", width=3)),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['val_acc'] * 100, name="Validation Accuracy", mode="lines+markers", line=dict(color="forestgreen", width=3)),
    secondary_y=True,
)

fig.update_layout(
    title_text="Training Loss and Validation Accuracy Over Epochs",
    xaxis_title="Epoch",
    template="plotly_dark",
    legend=dict(x=0.01, y=0.99)
)

fig.update_yaxes(title_text="Cross Entropy Loss", secondary_y=False)
fig.update_yaxes(title_text="Accuracy (%)", secondary_y=True)

fig.show()

## 2. Hardware Utilization (Peak VRAM Allocated vs. Reserved)

VRAM allocation tracking is crucial in LLM engineering. 
- **Allocated Memory**: The VRAM actively holding tensors.
- **Reserved Memory**: The VRAM cached by PyTorch's memory allocator (caching allocator) to avoid the high overhead of repeatedly querying the CUDA driver.

In [10]:
fig_vram = go.Figure()
fig_vram.add_trace(go.Bar(
    x=df['epoch'], 
    y=df['peak_vram_allocated_mb'],
    name='Peak VRAM Allocated (MB)',
    marker_color='crimson'
))
fig_vram.add_trace(go.Bar(
    x=df['epoch'], 
    y=df['peak_vram_reserved_mb'],
    name='Peak VRAM Reserved (MB)',
    marker_color='lightcoral'
))

fig_vram.update_layout(
    barmode='group',
    title_text='Peak GPU VRAM Usage (Allocated vs. Reserved)',
    xaxis_title='Epoch',
    yaxis_title='VRAM (MB)',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_vram.show()

## 3. Data Processing Rate (Throughput vs. Goodput)

- **Throughput**: Total raw tokens (characters) processed per second.
- **Goodput**: Useful tokens processed per second (excluding padding).

### Why do the Throughput and Goodput lines overlap exactly?
In this character-level implementation, we slice the text corpus into fixed chunks of 20 characters directly. **No padding tokens (`<pad>`) are used.** Since $100\%$ of the processed tokens are useful learning tokens, **Goodput is mathematically identical to Throughput**, resulting in the two lines overlapping perfectly on the chart.

In [11]:
fig_tp = go.Figure()
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['throughput_tokens_sec'],
    name='Throughput (Tokens/Sec)',
    mode='lines+markers',
    line=dict(color='darkorange', width=3)
))
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['goodput_tokens_sec'],
    name='Goodput (Tokens/Sec)',
    mode='lines+markers',
    line=dict(color='orange', width=2, dash='dash')
))

fig_tp.update_layout(
    title_text='Data Processing Rate (Throughput vs. Goodput)',
    xaxis_title='Epoch',
    yaxis_title='Tokens / Second',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_tp.show()

## 4. Compute Performance (TFLOPs and Model FLOPs Utilization)

- **Achieved TFLOPs/sec**: The absolute processing rate of raw floating point operations.
- **Model FLOPs Utilization (MFU %)**: The ratio of achieved compute performance to the hardware's peak theoretical performance. For your RTX 4070 Super, peak dense FP16 tensor core throughput is **142.2 TFLOPs/sec**.

In [12]:
# Plot Achieved TFLOPs/sec
fig_tflops = go.Figure()
fig_tflops.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['tflops_per_sec'], 
    name="Achieved TFLOPs/sec", 
    mode="lines+markers", 
    line=dict(color="mediumpurple", width=3)
))
fig_tflops.update_layout(
    title_text="Achieved Compute Performance (TFLOPs/sec)",
    xaxis_title="Epoch",
    yaxis_title="TFLOPs / Second",
    template="plotly_dark"
)
fig_tflops.show()

# Plot MFU (%)
fig_mfu = go.Figure()
fig_mfu.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['mfu_percent'], 
    name="MFU (%)", 
    mode="lines+markers", 
    line=dict(color="hotpink", width=3)
))
fig_mfu.update_layout(
    title_text="Model FLOPs Utilization (MFU %)",
    xaxis_title="Epoch",
    yaxis_title="MFU (%)",
    template="plotly_dark"
)
fig_mfu.show()

## 5. Roofline Model Analysis

A Roofline chart maps achieved performance (TFLOPs/sec) on the y-axis against the arithmetic intensity (FLOPs/Byte) on the x-axis. This highlights whether your training workload is memory-bandwidth bound or compute bound.

In [ ]:
# Plot Roofline chart
import numpy as np

# 1. Hardware Limits for RTX 4070 Super
peak_bandwidth_gb_s = 504.0
peak_fp16_tensor_tflops = 142.2
peak_fp32_vector_tflops = 35.5

# 2. Generate Roofline boundary lines
intensities = np.logspace(-3, 3, 500)
perf_fp16_ceiling = np.minimum(intensities * (peak_bandwidth_gb_s / 1000.0), peak_fp16_tensor_tflops)
perf_fp32_ceiling = np.minimum(intensities * (peak_bandwidth_gb_s / 1000.0), peak_fp32_vector_tflops)

# 3. Create the figure
fig_roofline = go.Figure()

fig_roofline.add_trace(go.Scatter(
    x=intensities,
    y=perf_fp16_ceiling,
    mode='lines',
    name='FP16 Tensor Core Peak (142.2 TFLOPs)',
    line=dict(color='crimson', width=3, dash='dash')
))

fig_roofline.add_trace(go.Scatter(
    x=intensities,
    y=perf_fp32_ceiling,
    mode='lines',
    name='FP32 Vector Peak (35.5 TFLOPs)',
    line=dict(color='orange', width=2, dash='dot')
))

# 4. Extract data points from training metrics
epochs = df['epoch'].tolist()
tflops_per_epoch = df['tflops_per_sec'].tolist()
intensities_per_epoch = df['arithmetic_intensity'].tolist()

fig_roofline.add_trace(go.Scatter(
    x=intensities_per_epoch,
    y=tflops_per_epoch,
    mode='markers+text',
    name='Your Model Training Run',
    text=[f"Epoch {e}" for e in epochs],
    textposition="top right",
    marker=dict(color='cyan', size=12, symbol='star', line=dict(color='white', width=1))
))

# 5. Format layout (log-log scale)
fig_roofline.update_layout(
    title=dict(text="NVIDIA GeForce RTX 4070 Super Roofline Analysis", font=dict(size=18)),
    xaxis=dict(
        title="Arithmetic Intensity (FLOPs / Byte)",
        type="log",
        gridcolor="#283442",
        zerolinecolor="#283442"
    ),
    yaxis=dict(
        title="Performance (TFLOPs / sec)",
        type="log",
        gridcolor="#283442",
        zerolinecolor="#283442",
        range=[-3, 2.5]
    ),
    template="plotly_dark",
    legend=dict(x=0.02, y=0.98),
    height=600
)
fig_roofline.show()